# 🚀 TARS — QLoRA Fine-Tuning + RAG Pipeline

```
Soru
  ↓
FAISS → docs/ + JSONL'den en alakalı 3 parça
  ↓
System prompt'a bağlam olarak enjekte et
  ↓
Fine-tune edilmiş Qwen cevap üretir
  ↓
Cevap + kaynak
```

### Adımlar
1. Kurulum
2. Veri yükleme (JSONL + docs)
3. RAG: chunk → embed → FAISS indeksi
4. Model + QLoRA kurulumu
5. Fine-tuning
6. RAG + Fine-tuned model birleşik sohbet

## 📦 1. Kurulum

In [ ]:
!pip install -q transformers peft accelerate datasets bitsandbytes trl
!pip install -q sentence-transformers faiss-cpu
!pip install -q python-docx pypdf
print('✅ Kurulum tamamlandı')

In [ ]:
import torch, json, os, glob
import numpy as np
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  GPU bulunamadı — CPU ile devam edilecek')

## ⚙️ 2. Ayarlar

Dosyalarınızı yüklemek için sol panelden **Files (📁)** ikonuna tıklayın:
- JSONL dosyaları → `/content/jsonl/` klasörüne
- PDF / DOCX dosyaları → `/content/docs/` klasörüne

In [ ]:
# ── Dizinler ──────────────────────────────────────────────────────────────
JSONL_DIR   = '/content/jsonl'
DOCS_DIR    = '/content/docs'
OUTPUT_DIR  = '/content/tars_lora'

# ── Model ─────────────────────────────────────────────────────────────────
LLM_MODEL   = 'Qwen/Qwen2.5-0.5B-Instruct'   # Üretici dil modeli
EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'  # RAG için embedding

# ── RAG ───────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 300   # Her parça kaç karakter
CHUNK_OVERLAP = 60    # Parçalar arası örtüşme
TOP_K         = 3     # Kaç parça getirilsin

# ── Fine-tuning ───────────────────────────────────────────────────────────
MAX_LENGTH  = 512
BATCH_SIZE  = 4
GRAD_ACCUM  = 4       # Efektif batch = 4×4 = 16
EPOCHS      = 3
LR          = 2e-4
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROP   = 0.05

os.makedirs(JSONL_DIR,  exist_ok=True)
os.makedirs(DOCS_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Dizinler hazır')

## 📂 3. Veri Yükleme

In [ ]:
# ── JSONL yükle ───────────────────────────────────────────────────────────
# Her satır: {"instruction": "...", "input": "...", "output": "..."}

raw_data = []
jsonl_files = list(Path(JSONL_DIR).glob('*.jsonl'))

if not jsonl_files:
    print('⚠️  JSONL bulunamadı — demo veri oluşturuluyor...')
    demo = [
        {"instruction": "ADC biriminin roket aviyonik sistemindeki rolü nedir?",
         "input": "Veri dönüştürme",
         "output": "Analog sensörlerden gelen voltaj değerlerini uçuş bilgisayarının işleyebileceği dijital verilere dönüştürür."},
        {"instruction": "IMU sensörünün uçuş kontrol sistemindeki işlevi nedir?",
         "input": "",
         "output": "İvme, açısal hız ve manyetik alan verilerini ölçerek roketin oryantasyonunu belirler."},
        {"instruction": "UART protokolü nedir?", "input": "Haberleşme",
         "output": "Asenkron seri haberleşme protokolüdür. TX ve RX hatları üzerinden veri iletir."},
        {"instruction": "SPI ile I2C arasındaki fark nedir?", "input": "",
         "output": "SPI 4 hat kullanır ve daha hızlıdır. I2C 2 hat kullanır ve daha az pin tüketir."},
        {"instruction": "PWM sinyali ne için kullanılır?", "input": "Motor kontrol",
         "output": "Motor hızı, servo konumu veya LED parlaklığını darbe genişliği ile kontrol eder."},
        {"instruction": "Watchdog timer nedir?", "input": "",
         "output": "Sistemin donmasını önlemek için belirli aralıklarla sıfırlanması gereken donanım sayacıdır."},
        {"instruction": "DMA nedir ve ne zaman kullanılır?", "input": "Bellek",
         "output": "CPU müdahalesi olmadan bellek ile çevre birimler arasında veri aktarımı sağlar."},
        {"instruction": "GPIO pini nasıl yapılandırılır?", "input": "",
         "output": "Input veya output olarak ayarlanır. Pull-up/pull-down dirençleri varsayılan durumu belirler."},
        {"instruction": "CAN bus neden tercih edilir?", "input": "Otomotiv",
         "output": "Gürültüye dayanıklılığı ve çok düğümlü yapısıyla endüstriyel uygulamalarda yaygın kullanılır."},
        {"instruction": "RTOS kullanmanın avantajları nelerdir?", "input": "",
         "output": "Gerçek zamanlı görev planlaması ve deterministik yanıt süresi sağlar."},
    ]
    with open(f'{JSONL_DIR}/demo.jsonl', 'w', encoding='utf-8') as f:
        for item in demo:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    jsonl_files = [Path(f'{JSONL_DIR}/demo.jsonl')]

for path in jsonl_files:
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    raw_data.append(json.loads(line))
                except:
                    pass

print(f'✅ JSONL kayıt: {len(raw_data)}')

In [ ]:
# ── Doküman yükle (PDF + DOCX) ────────────────────────────────────────────
# Bu belgeler RAG için kullanılacak — fine-tuning'e girmeyecek

def read_pdf(path):
    """PDF'den metin çıkar."""
    try:
        from pypdf import PdfReader
        reader = PdfReader(path)
        return '\n'.join(p.extract_text() or '' for p in reader.pages)
    except Exception as e:
        print(f'  ⚠️  PDF okuma hatası ({Path(path).name}): {e}')
        return ''

def read_docx(path):
    """Word dosyasından metin çıkar."""
    try:
        from docx import Document
        doc = Document(path)
        return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())
    except Exception as e:
        print(f'  ⚠️  DOCX okuma hatası ({Path(path).name}): {e}')
        return ''

documents = []  # [{"title": ..., "text": ..., "source": ...}]

for path in glob.glob(f'{DOCS_DIR}/**/*.pdf', recursive=True):
    text = read_pdf(path)
    if text.strip():
        documents.append({'title': Path(path).stem, 'text': text, 'source': Path(path).name})
        print(f'  📄 PDF:  {Path(path).name} ({len(text)} karakter)')

for path in glob.glob(f'{DOCS_DIR}/**/*.docx', recursive=True):
    text = read_docx(path)
    if text.strip():
        documents.append({'title': Path(path).stem, 'text': text, 'source': Path(path).name})
        print(f'  📄 DOCX: {Path(path).name} ({len(text)} karakter)')

# JSONL verisi de RAG havuzuna giriyor
# Böylece model hem dokümanları hem de soru-cevap çiftlerini arayabilir
for item in raw_data:
    full_text = f"Soru: {item['instruction']}\nCevap: {item['output']}"
    documents.append({
        'title': item['instruction'][:60],
        'text':  full_text,
        'source': 'jsonl'
    })

print(f'\n✅ Toplam doküman: {len(documents)} '
      f'(PDF/DOCX: {len(documents)-len(raw_data)}, JSONL: {len(raw_data)})')

## 🔍 4. RAG — Chunk → Embed → FAISS

Bu bölüm **eğitimden bağımsız** çalışır.  
Dokümanları parçalayıp vektöre çevirir ve FAISS indeksine kaydeder.  
Inference sırasında soruya en yakın parçalar bulunup modele bağlam olarak verilir.

In [ ]:
# ── 4a. Metin parçalama ───────────────────────────────────────────────────
# Neden chunk'lara bölüyoruz?
# Tüm dokümanı modele veremeyiz — 512 token limitimiz var.
# Sadece soruyla alakalı kısımları çekip göndermek daha verimli.

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Metni örtüşen parçalara böl."""
    chunks = []
    start  = 0
    while start < len(text):
        end   = min(start + size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        # Bir sonraki parça overlap kadar geri başlar
        # Böylece cümle ortasında kesilen bilgi kaybolmaz
        start += size - overlap
    return chunks

all_chunks = []  # [{"text": ..., "source": ..., "title": ...}]

for doc in documents:
    for chunk in chunk_text(doc['text']):
        all_chunks.append({
            'text':   chunk,
            'source': doc['source'],
            'title':  doc['title'],
        })

print(f'✅ Toplam chunk: {len(all_chunks)}')
print(f'   Ort. chunk uzunluğu: {np.mean([len(c["text"]) for c in all_chunks]):.0f} karakter')

In [ ]:
# ── 4b. Embedding ─────────────────────────────────────────────────────────
# Neden ayrı bir embedding modeli?
# Qwen metin üretmek için, MiniLM benzerlik aramak için optimize.
# İki farklı iş, iki farklı model.
#
# paraphrase-multilingual-MiniLM-L12-v2:
# - 50+ dil destekliyor (Türkçe dahil)
# - 384 boyutlu vektör üretiyor
# - Sadece ~120 MB — hızlı yüklenir

from sentence_transformers import SentenceTransformer

print(f'Embedding modeli yükleniyor: {EMBED_MODEL}')
embed_model = SentenceTransformer(EMBED_MODEL)

print('Chunk\'lar vektöre çevriliyor...')
chunk_texts = [c['text'] for c in all_chunks]
embeddings  = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,   # Kosinüs benzerliği için normalize et
    show_progress_bar=True,
    batch_size=64,
)
embeddings = np.array(embeddings, dtype=np.float32)
print(f'✅ Embedding matrisi: {embeddings.shape}')

In [ ]:
# ── 4c. FAISS indeksi ─────────────────────────────────────────────────────
# Neden FAISS?
# Binlerce vektör içinde en yakın k tanesini milisaniyeler içinde buluyor.
# IndexFlatIP: normalize vektörlerde iç çarpım = kosinüs benzerliği.

import faiss

EMBED_DIM = embeddings.shape[1]   # 384
index     = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

print(f'✅ FAISS indeksi: {index.ntotal} vektör ({EMBED_DIM} boyut)')

def retrieve(query, k=TOP_K):
    """
    Verilen soruya en benzer k chunk'ı döndür.
    Dönen liste: [{"text": ..., "source": ..., "score": ...}]
    """
    # Soruyu da aynı embedding modeliyle vektöre çevir
    q_emb = embed_model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = index.search(q_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'text':   all_chunks[idx]['text'],
            'source': all_chunks[idx]['source'],
            'title':  all_chunks[idx]['title'],
            'score':  float(score),
        })
    return results

# Hızlı test
test_results = retrieve('ADC birimi ne işe yarar?')
print('\n── Retrieval testi ──')
for r in test_results:
    print(f'  [{r["score"]:.3f}] {r["source"]} → {r["text"][:80]}...')

## 🤖 5. QLoRA Fine-Tuning

JSONL verisiyle modeli ince ayara tabi tutuyoruz.  
**Bu bölüm RAG'dan bağımsız** — iki bileşen ayrı eğitilir, inference'da birleşir.

In [ ]:
# ── 5a. Veri formatlama ───────────────────────────────────────────────────
# Alpaca formatını Qwen ChatML formatına çevir

SYSTEM_PROMPT = (
    'Sen TARS adlı teknik asistansın. '
    'Gömülü sistemler, roket aviyoniği ve elektronik konularında uzman bir yardımcısın. '
    'Soruları açık, teknik ve Türkçe olarak yanıtla.'
)

def format_sample(item):
    instruction = item.get('instruction', '').strip()
    inp         = item.get('input', '').strip()
    output      = item.get('output', '').strip()
    user_msg    = f'{instruction}\nBağlam: {inp}' if inp else instruction
    return {'text': (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_msg}<|im_end|>\n'
        f'<|im_start|>assistant\n{output}<|im_end|>'
    )}

from datasets import Dataset

formatted = [format_sample(d) for d in raw_data]
np.random.seed(42)
idx   = np.random.permutation(len(formatted))
split = max(1, int(len(formatted) * 0.9))

train_dataset = Dataset.from_list([formatted[i] for i in idx[:split]])
val_dataset   = Dataset.from_list([formatted[i] for i in idx[split:]])
print(f'✅ Train: {len(train_dataset)}  |  Val: {len(val_dataset)}')

In [ ]:
# ── 5b. Model yükle (4-bit QLoRA) ────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f'Model yükleniyor: {LLM_MODEL}')

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','v_proj','k_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=LORA_DROP, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 5c. Eğitim ────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    fp16                        = True,
    logging_steps               = 5,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',
    max_seq_length              = MAX_LENGTH,
    dataset_text_field          = 'text',
    packing                     = False,
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print('🚀 Eğitim başlıyor...')
train_result = trainer.train()
print(f'\n✅ Eğitim tamamlandı! Loss: {train_result.training_loss:.4f}')

In [ ]:
# ── 5d. Adaptörü kaydet + loss grafiği ───────────────────────────────────
adapter_path = f'{OUTPUT_DIR}/final_adapter'
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adaptör kaydedildi: {adapter_path}')

import matplotlib.pyplot as plt

train_logs = [x for x in trainer.state.log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs  = [x for x in trainer.state.log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
if train_logs:
    ax.plot([x['step'] for x in train_logs], [x['loss'] for x in train_logs],
            label='Train loss', color='#0F766E', lw=2)
if eval_logs:
    ax.plot([x['step'] for x in eval_logs], [x['eval_loss'] for x in eval_logs],
            label='Val loss', color='#DC2626', lw=2, marker='o')
ax.set_xlabel('Adım'); ax.set_ylabel('Loss')
ax.set_title('TARS QLoRA — Eğitim Kaybı', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=150); plt.show()

## 💬 6. RAG + Fine-Tuned Model — Birleşik Sohbet

Artık iki bileşen bir arada çalışıyor:

```
Soru
 ├─► FAISS → top-3 chunk (bağlam)
 │
 └─► System prompt'a bağlam enjekte et
         ↓
     Fine-tuned Qwen → Türkçe cevap
```

In [ ]:
# ── 6a. Inference modeli yükle ────────────────────────────────────────────
# Eğitimden hemen sonra çalıştırıyorsanız bu cell'i ATLAYIN
# (model zaten bellekte)

from peft import PeftModel

adapter_path = f'{OUTPUT_DIR}/final_adapter'
print('Model yükleniyor...')

inf_tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
inf_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True,
)
inf_model = PeftModel.from_pretrained(inf_model, adapter_path)
inf_model.eval()
print('✅ Model hazır')

In [ ]:
# ── 6b. RAG + LLM birleşik fonksiyon ─────────────────────────────────────

def tars_rag_chat(soru, max_new_tokens=256, temperature=0.3, verbose=True):
    """
    1. FAISS ile ilgili chunk'ları bul
    2. Chunk'ları sistem mesajına bağlam olarak ekle
    3. Fine-tune edilmiş model cevap üretir
    """

    # ── Adım 1: Retrieve ──────────────────────────────────────────────────
    retrieved = retrieve(soru, k=TOP_K)

    if verbose:
        print('\n── Bulunan kaynaklar ──')
        for r in retrieved:
            print(f'  [{r["score"]:.3f}] {r["source"]} → {r["text"][:70]}...')

    # ── Adım 2: Bağlamı oluştur ───────────────────────────────────────────
    # Bulunan chunk'lar sistem mesajına ekleniyor
    # Model "şu kaynaklara göre cevapla" talimatını alıyor
    context_parts = []
    for i, r in enumerate(retrieved, 1):
        context_parts.append(f'[Kaynak {i} — {r["source"]}]\n{r["text"]}')
    context = '\n\n'.join(context_parts)

    system_with_context = (
        f'Sen TARS adlı teknik asistansın. '
        f'Gömülü sistemler, roket aviyoniği ve elektronik konularında uzman bir yardımcısın. '
        f'Soruları açık, teknik ve Türkçe olarak yanıtla.\n\n'
        f'Aşağıdaki kaynaklardan yararlanarak cevap ver. '
        f'Kaynakta olmayan bilgileri ekleme.\n\n'
        f'{context}'
    )

    # ── Adım 3: Prompt oluştur ────────────────────────────────────────────
    prompt = (
        f'<|im_start|>system\n{system_with_context}<|im_end|>\n'
        f'<|im_start|>user\n{soru}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

    # ── Adım 4: Üretim ────────────────────────────────────────────────────
    inputs = inf_tokenizer(prompt, return_tensors='pt').to(inf_model.device)

    # MAX_LENGTH aşılıyorsa uyar
    if inputs['input_ids'].shape[1] > MAX_LENGTH - 50:
        print(f'  ⚠️  Prompt uzun ({inputs["input_ids"].shape[1]} token), bağlam kısaltıldı.')

    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            do_sample          = temperature > 0,
            repetition_penalty = 1.1,
            eos_token_id       = inf_tokenizer.convert_tokens_to_ids('<|im_end|>'),
            pad_token_id       = inf_tokenizer.pad_token_id,
        )

    # Sadece yeni üretilen kısım
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer = inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer, retrieved


print('✅ tars_rag_chat() hazır')

In [ ]:
# ── 6c. Test soruları ─────────────────────────────────────────────────────

test_questions = [
    'ADC biriminin roket aviyonik sistemindeki rolü nedir?',
    'IMU sensörü hangi verileri ölçer?',
    'UART ile SPI arasındaki temel fark nedir?',
    'Watchdog timer neden kullanılır?',
]

print('=' * 65)
print('  TARS — RAG + Fine-Tuned Model Test')
print('=' * 65)

for soru in test_questions:
    print(f'\n❓ {soru}')
    cevap, kaynaklar = tars_rag_chat(soru, verbose=True)
    print(f'\n🤖 {cevap}')
    print(f'\n📚 Kaynaklar: {", ".join(set(r["source"] for r in kaynaklar))}')
    print('─' * 55)

In [ ]:
# ── 6d. İnteraktif sohbet ─────────────────────────────────────────────────

print('TARS hazır. Sorunuzu yazın (çıkmak için q)\n')

while True:
    soru = input('Siz: ').strip()
    if soru.lower() in ('q', 'quit', 'exit', ''):
        print('Görüşmek üzere!')
        break

    cevap, kaynaklar = tars_rag_chat(soru, verbose=False)

    print(f'\nTARS: {cevap}')
    print(f'Kaynaklar: {", ".join(set(r["source"] for r in kaynaklar))}')
    print(f'Retrieval skorları: {[round(r["score"],3) for r in kaynaklar]}')
    print()

## 💾 7. Google Drive'a Kaydet (Opsiyonel)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = '/content/drive/MyDrive/TARS'
shutil.copytree(OUTPUT_DIR, drive_path, dirs_exist_ok=True)
print(f'✅ {drive_path} adresine kaydedildi')

---
## 📋 Pipeline Özeti

```
docs/ (PDF, DOCX)
jsonl/ (instruction/output)  ──► chunk ──► MiniLM embed ──► FAISS indeksi
                                                                   │
jsonl/ (instruction/output)  ──► ChatML format ──► QLoRA FT       │
                                                       │           │
                                              Fine-tuned Qwen      │
                                                       │           │
Kullanıcı sorusu ─────────────────────────────────────┴───────────┘
                                                       │
                              bağlam enjeksiyonu + cevap üretimi
                                                       │
                                              Türkçe teknik cevap
                                              + kaynak referansı
```

| Bileşen | Model | Görev |
|---------|-------|-------|
| Embedding | MiniLM-L12 (384d) | Chunk benzerlik araması |
| Fine-tuning | Qwen2.5-0.5B (QLoRA r=16) | Türkçe teknik cevap |
| Vektör DB | FAISS IndexFlatIP | Top-k retrieval |